# STEP 8 - one identical protocol for every encoder

Until now the models were trained differently: XLM-R, BanglaBERT and MuRIL on the full
40,224 comments at batch 32 in fp16, but BanglaT5 on a 20,000 subsample at batch 16 in
fp32. That lets a reviewer answer our central claim ("same lab, only the tokenizer
differs") with "your training setup differs too".

This notebook removes that objection. **Every model, identical protocol:**
full training set, batch 32, learning rate 2e-5, 2 epochs, max length 128, seed 42,
**fp32 throughout** (fp16 is what breaks mDeBERTa-v3, so nobody gets fp16).

mDeBERTa-v3 previously died with a non-finite loss. Three fixes here: fp32, a lower
learning rate (1e-5, its documented requirement), and a longer warmup. If it still
produces a non-finite loss the notebook retries once at 5e-6 before giving up.

It also extends the tokenizer survey with Gemma, Llama, Mistral and others. That part
needs no GPU and no training.

**Settings:** GPU **T4 x2**, Internet **ON**, data **`naurosromim/bdshs`**.
Run with **Save Version -> Save & Run All (Commit)**. Budget **about 4 hours** - fp32 is
roughly 2.5x slower than fp16. Results are written after every model, so a timeout loses
at most one model.


In [ ]:
!pip install -q "emoji==1.7.0" regex ftfy uharfbuzz sentencepiece protobuf tiktoken
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
import sys, os, glob, time, random, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd, torch
from functools import lru_cache
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

DEV, OUT = "cuda", "/kaggle/working"
R = ra.Renderer(ra.fetch_fonts(f"{OUT}/fonts"))
print("GPU:", torch.cuda.get_device_name(0), "| fonts:", len(R.names))

paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
rtr, rte = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in rtr.columns if rtr[c].dtype == object]
TEXT = max(obj, key=lambda c: rtr[c].astype(str).str.len().mean())
LABEL = [c for c in rtr.columns if c != TEXT and rtr[c].dropna().nunique() == 2][0]
def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)
train, test = prep(rtr), prep(rte)
HATE = 1
print("train", len(train), "(full) | test", len(test))

# identical attack sets, identical seeds as Step 6
N_ATK = 1500
rng = random.Random(0)
hate_txt = [test.text[i] for i in sorted(rng.sample(test.index[test.label == HATE].tolist(), N_ATK))]
ben_txt  = [test.text[i] for i in sorted(rng.sample(test.index[test.label != HATE].tolist(), N_ATK))]
ATK = {
    "gated_inert_ALL":           [R.attack(t, "ALL", mode="inert", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_adaptive_hasant_ALL": [R.attack(t, "ALL", mode="hasant", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_both_ALL":            [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(hate_txt)],
}
BEN_ATK = [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(ben_txt)]
DEFENSES = {"none": lambda t: t, "nfc": ra.defend_nfc, "strip_all": ra.defend_strip_all,
            "position_aware": ra.defend_position_aware, "render_equivalence": R.defend_render_equivalence}
DEFENDED = {(a, d): [fn(t) for t in ATK[a]] for a in ATK for d, fn in DEFENSES.items()}
CLEAN_DEF = {d: [fn(t) for t in test.text] for d, fn in DEFENSES.items()}
for k, v in ATK.items():
    print(f"  {k:26} visible in any font: {np.mean([len(R.visible_in(c,a))>0 for c,a in zip(hate_txt,v)]):.4f}")

In [ ]:
# PART 1 - extended tokenizer survey. No GPU, no training.
import tiktoken
CAND = [
    # WordPiece / BertTokenizer
    ("csebuetnlp/banglabert", "WordPiece"), ("csebuetnlp/banglishbert", "WordPiece"),
    ("sagorsarker/bangla-bert-base", "WordPiece"), ("google/muril-base-cased", "WordPiece"),
    ("bert-base-multilingual-cased", "WordPiece"),
    # SentencePiece
    ("xlm-roberta-base", "SentencePiece"), ("csebuetnlp/banglat5", "SentencePiece"),
    ("google/mt5-base", "SentencePiece"), ("facebook/mbart-large-50", "SentencePiece"),
    ("microsoft/mdeberta-v3-base", "SentencePiece"), ("ai4bharat/indic-bert", "SentencePiece"),
    ("google/gemma-2-2b", "SentencePiece"), ("google/gemma-3-4b-it", "SentencePiece"),
    ("mistralai/Mistral-7B-v0.3", "SentencePiece"),
    # BPE
    ("Qwen/Qwen2.5-7B-Instruct", "BPE"), ("meta-llama/Llama-3.1-8B", "BPE"),
    ("deepseek-ai/DeepSeek-V3", "BPE"),
]
probe = hate_txt[:200]
probe_atk = [ATK["gated_inert_ALL"][i] for i in range(200)]
rows = []
for name, fam in CAND:
    try:
        tk = AutoTokenizer.from_pretrained(name)
        cl = float(np.mean([len(tk.tokenize(normalize(t))) for t in probe]))
        at = float(np.mean([len(tk.tokenize(normalize(t))) for t in probe_atk]))
        rows.append({"model": name, "family": fam, "tokenizer": type(tk).__name__,
                     "vulnerable": at != cl, "clean_tokens": round(cl, 1),
                     "attacked_tokens": round(at, 1), "inflation_pct": round((at/cl-1)*100, 1)})
        print(f"  {name:34} {type(tk).__name__:24} inflation {(at/cl-1)*100:+7.1f}%")
    except Exception as ex:
        rows.append({"model": name, "family": fam, "tokenizer": "LOAD_FAIL",
                     "vulnerable": None, "clean_tokens": None, "attacked_tokens": None,
                     "inflation_pct": None})
        print(f"  {name:34} LOAD_FAIL ({type(ex).__name__}) - gated or unavailable")
for enc in ["o200k_base", "cl100k_base"]:
    e = tiktoken.get_encoding(enc)
    cl = float(np.mean([len(e.encode(t)) for t in probe]))
    at = float(np.mean([len(e.encode(t)) for t in probe_atk]))
    rows.append({"model": f"tiktoken/{enc}", "family": "BPE", "tokenizer": "tiktoken",
                 "vulnerable": at != cl, "clean_tokens": round(cl,1),
                 "attacked_tokens": round(at,1), "inflation_pct": round((at/cl-1)*100,1)})
    print(f"  tiktoken/{enc:26} inflation {(at/cl-1)*100:+7.1f}%")
surv = pd.DataFrame(rows); surv.to_csv(f"{OUT}/step8_tokenizer_survey.csv", index=False)
print("\nloaded:", int((surv.tokenizer != "LOAD_FAIL").sum()), "of", len(surv))

In [ ]:
# PART 2 - ONE protocol for every encoder. fp32 for all, no exceptions.
PROTOCOL = dict(train_size=len(train), batch=32, epochs=2, maxlen=128, seed=42, precision="fp32")
print("protocol:", PROTOCOL)

@lru_cache(maxsize=None)
def norm(t): return normalize(t)

def train_one(name, lr, warmup=0.1, seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tk = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).to(DEV)
    if getattr(mdl.config, "pad_token_id", None) is None:
        mdl.config.pad_token_id = tk.pad_token_id
    e = tk([norm(t) for t in train.text], truncation=True, padding="max_length",
           max_length=PROTOCOL["maxlen"], return_tensors="pt")
    y = torch.tensor(train.label.values)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(TensorDataset(e["input_ids"], e["attention_mask"], y),
                    batch_size=PROTOCOL["batch"], shuffle=True, generator=g)
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
             total_steps=PROTOCOL["epochs"]*len(dl), pct_start=warmup)
    mdl.train()
    for ep in range(PROTOCOL["epochs"]):
        for st, (i_, m_, l_) in enumerate(dl):
            i_, m_, l_ = i_.to(DEV), m_.to(DEV), l_.to(DEV)
            opt.zero_grad()
            loss = mdl(input_ids=i_, attention_mask=m_, labels=l_).loss    # fp32, no autocast
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite loss at epoch {ep} step {st}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step(); sch.step()
            if st % 500 == 0: print(f"    ep{ep} {st}/{len(dl)} loss {loss.item():.4f}")
    mdl.eval()
    return tk, mdl

def evaluate(name, family, tk, mdl):
    @torch.no_grad()
    def pred(texts, bs=64):
        ids = tk([norm(t) for t in texts], truncation=True, max_length=512)["input_ids"]
        order = np.argsort([len(z) for z in ids]); out = np.empty(len(texts), dtype=int)
        for s in range(0, len(order), bs):
            idx = order[s:s+bs]
            b = tk.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
            lg = mdl(input_ids=b["input_ids"].to(DEV), attention_mask=b["attention_mask"].to(DEV)).logits
            out[idx] = lg.float().argmax(-1).cpu().numpy()
        return out
    ty = test.label.values
    clean = [{"model": name, "family": family, "defense_on_clean": d,
              "acc": round(accuracy_score(ty, p), 4), "f1": round(f1_score(ty, p), 4)}
             for d in DEFENSES for p in [pred(CLEAN_DEF[d])]]
    hp = pred(hate_txt); tgt = np.where(hp == HATE)[0]
    lad = [{"model": name, "family": family, "tokenizer": type(tk).__name__, "config": cfg,
            "n_targets": len(tgt), "evasion": round(float((pred([ATK[cfg][i] for i in tgt]) != HATE).mean()), 4)}
           for cfg in ATK]
    dfn = [{"model": name, "attack": a, "defense": d,
            "evasion": round(float((pred([DEFENDED[(a, d)][i] for i in tgt]) != HATE).mean()), 4)}
           for a in ATK for d in DEFENSES]
    bp, bpa = pred(ben_txt), pred(BEN_ATK)
    ben = {"model": name, "family": family,
           "benign_fpr_clean": round(float((bp == HATE).mean()), 4),
           "benign_fpr_attacked": round(float((bpa == HATE).mean()), 4),
           "benign_flip_rate": round(float((bp != bpa).mean()), 4)}
    for r in lad: print("   ", r)
    return clean, lad, dfn, ben

In [ ]:
# PART 3 - run every model. mDeBERTa-v3 gets its documented lower learning rate.
MODELS = [
    ("csebuetnlp/banglabert",      "WordPiece",     2e-5),
    ("google/muril-base-cased",    "WordPiece",     2e-5),
    ("xlm-roberta-base",           "SentencePiece", 2e-5),
    ("csebuetnlp/banglat5",        "SentencePiece", 2e-5),
    ("microsoft/mdeberta-v3-base", "SentencePiece", 1e-5),   # fp32 + low lr + long warmup
]
CL, LA, DE, BE, ST = [], [], [], [], []
for name, fam, lr in MODELS:
    t0 = time.time(); done = False
    for attempt_lr, wu in [(lr, 0.2), (5e-6, 0.3)]:
        try:
            tk, mdl = train_one(name, attempt_lr, warmup=wu)
            c, l, d, b = evaluate(name, fam, tk, mdl)
            CL += c; LA += l; DE += d; BE.append(b)
            ST.append({"model": name, "family": fam, "ok": True, "lr": attempt_lr,
                       "warmup": wu, "note": "" if attempt_lr == lr else "retried at lower lr"})
            del mdl; torch.cuda.empty_cache(); done = True
            print(f"== {name} OK at lr={attempt_lr} in {(time.time()-t0)/60:.1f} min\n")
            break
        except Exception as ex:
            print(f"   {name} failed at lr={attempt_lr}: {type(ex).__name__}: {str(ex)[:110]}")
            torch.cuda.empty_cache()
    if not done:
        ST.append({"model": name, "family": fam, "ok": False, "lr": None, "warmup": None,
                   "note": "failed at both learning rates"})
    for df, fn in [(CL,"clean_cost"), (LA,"ladder"), (DE,"defense"), (BE,"benign"), (ST,"status")]:
        pd.DataFrame(df).to_csv(f"{OUT}/step8_{fn}.csv", index=False)
pd.DataFrame([PROTOCOL]).to_csv(f"{OUT}/step8_protocol.csv", index=False)

In [ ]:
pd.set_option("display.width", 250)
print("="*94); print("STEP 8 SUMMARY - one identical protocol"); print("="*94)
print("protocol:", PROTOCOL)
print("\n-- which models trained --"); print(pd.DataFrame(ST).to_string(index=False))
print("\n-- evasion, all models, same protocol --"); print(pd.DataFrame(LA).to_string(index=False))
print("\n-- clean accuracy --")
print(pd.DataFrame(CL).query("defense_on_clean=='none'").to_string(index=False))
print("\n-- defenses --"); print(pd.DataFrame(DE).to_string(index=False))
print("\n-- benign control --"); print(pd.DataFrame(BE).to_string(index=False))
print("\n-- extended tokenizer survey --")
print(surv.to_string(index=False))
print("="*94)